# Feature Engineering

In real projects, data rarely arrives in a model-ready format. That is why **feature engineering** is a core part of machine learning work.

Feature engineering helps you:
- make data usable by models,
- reduce data quality issues before training,
- and improve reproducibility when new data arrives.

Examples:
- Many models cannot handle missing values directly.
- Some algorithms are sensitive to feature scale.
- Categorical values usually need explicit encoding.

Also remember: not every transformation helps every model. For example, tree-based models are usually scale-invariant, while distance-based models often benefit from scaling.

The goal is not to apply every technique. The goal is to apply the **right** transformations for the data and model you are using.


> **Checkpoint:**
> You can explain why preprocessing decisions depend on both data quality and model type.

> **Common pitfalls:**
> - Applying transformations without checking model assumptions.
> - One-hot encoding very high-cardinality columns without evaluating sparsity impact.
> - Treating exploration code as production-ready preprocessing.

> **Self-check:**
> If new data arrives tomorrow, can you apply the same transformations consistently?


## Visual Guide: Feature Engineering Flow

```mermaid
flowchart TD
    A["Raw Seattle weather data"]
    B["Validate and inspect columns"]
    C["Clean messy feature values"]
    D["Handle missing values"]
    E["Encode or scale features"]
    F["Reusable train/new-ready dataset"]

    A --> B --> C --> D --> E --> F
```


## Loading data and preparation steps
We will use the **Seattle Weather Dataset** as an example in this notebook and since we already learned Pydantic we can use it to validate if our data looks as expected. Let's import the data and write our **Pydantic model**!


In [ ]:
# pandas is our main tool for tabular data manipulation in this notebook.
import pandas as pd

# NumPy gives us numerical helpers such as square root for transformations.
import numpy as np

# We expect the `date` column to become real Python datetime values.
from datetime import datetime
from pydantic import BaseModel

# We split the data to simulate a train set and future unseen data.
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# For Q-Q plots
import scipy.stats as stats

# `re` helps us clean inconsistent weather labels with regular expressions.
import re

In [ ]:
# Import the data
seattle_weather_data = pd.read_csv("../data/seattle-weather_raw.csv")
seattle_weather_data.head()

Let's see what columns we have in this DataFrame and think about which **data types** we would expect:

| Column | Expected data type |
|---|---|
| `date` | datetime |
| `precipitation` | float |
| `temp_max` | float |
| `temp_min` | float |
| `wind` | float |
| `weather` | string |

We can use this now for our Pydantic model:


In [ ]:
class DataValidation(BaseModel):
    date: datetime
    precipitation: float
    temp_max: float
    temp_min: float
    wind: float
    weather: str

We can **test** if it works as expected on a simple dictionary of **one of the observations**:


In [ ]:
# A single-row example helps us test the validation schema before applying it to the whole DataFrame.
sample_record = {
    "date": "2012/01/01",
    "precipitation": 0.0,
    "temp_max": 12.8,
    "temp_min": "41.0 F",
    "wind": 4.710858197573892,
    "weather": "drizzle",
}

try:
    # This should fail because `temp_min` is still a string, not a float.
    DataValidation.model_validate(sample_record)
except Exception as error:
    print(error)

Of course it will fail because our data does not have the correct types (just look at the `temp_min` column).

To be able to apply the Pydantic model to our pandas DataFrame and validate the types, we need to define a little helper class, because the ```DataValidation``` class only accepts dictionaries.  

We have to **transform the pandas DataFrame into a dictionary**.


In [ ]:
class DataFrameValidation(BaseModel):
    # The DataFrame will be represented as a list of row dictionaries.
    df_as_dict: list[DataValidation]


# Transform the DataFrame into normal Python dictionaries because
# Pydantic validates Python objects, not pandas DataFrames directly.
data_dict = seattle_weather_data.to_dict(orient="records")

try:
    # This validates every row against the `DataValidation` schema.
    DataFrameValidation.model_validate({"df_as_dict": data_dict})
except Exception as error:
    print(error)

Let's transform this code into a **validation function**:


In [ ]:
def data_validation(df: pd.DataFrame, data_schema) -> pd.DataFrame:
    class DataFrameValidation(BaseModel):
        # Build a list-of-rows schema dynamically from the model passed in.
        df_as_dict: list[data_schema]

    # Convert the DataFrame into records so each row can be validated separately.
    df_as_dict = df.to_dict(orient="records")

    # If a row has the wrong type, Pydantic will raise an error here.
    DataFrameValidation.model_validate({"df_as_dict": df_as_dict})

    # Returning the original DataFrame makes this helper easy to reuse later.
    return df

Now it's time to start with the actual **feature engineering**!


## Dealing with messy data

Before we start to look at missing values, scale the data or think about how to handle categories, we'll focus on **changing the features to make more sense**.  

Look at the `temp_min` column. We would expect it to be **floats**, but the column contains **strings** because the entries include an `F`. Based on the `F` and the (on first glance) weird values, we can assume that this column is in Fahrenheit and not in Celsius.  

We'll start by **converting the entries from strings to floats and from Fahrenheit to Celsius**.


In [ ]:
# Start from a copy so the original raw dataset stays available for comparison.
seattle_weather_data_pandas = seattle_weather_data.copy()

In [ ]:
# `temp_min` currently contains strings like `41.0 F`.
# We remove the trailing ` F` and then cast the cleaned strings to floats.
seattle_weather_data_pandas["temp_min"] = (
    seattle_weather_data_pandas.temp_min.str.strip(" F").astype("float")
)

# Check that the dtype is now numeric.
seattle_weather_data_pandas.temp_min.dtype

Now that we have floats in our column we can **convert the temperature** from Fahrenheit to Celsius.


In [ ]:
# Convert `temp_min` from Fahrenheit to Celsius.
# We apply the formula to every value in the column.
seattle_weather_data_pandas["temp_min"] = seattle_weather_data_pandas["temp_min"].apply(
    lambda x: (x - 32) / 1.8
)
seattle_weather_data_pandas.head(5)

In the next step we'll deal with the `weather` column:


In [ ]:
# Inspect the unique category values before cleaning them.
# This tells us which abbreviations, punctuation variants, and capitalizations exist.
seattle_weather_data_pandas["weather"].unique()

A look at the **unique values** shows that "drizzle", "d", "driz." and "Drizzle" are probably all the same category.  

Seems like one category can have several values. This often happens if there is no fixed standard for data collection. We need to **reduce the number of values for each category to one**. In our case we will create a dictionary and match the patterns with [regex](https://docs.python.org/3/library/re.html).  

> **Note:** Sometimes it can happen that **abbreviations are ambiguous**. For example, there is a category "sn" which could represent either "snow" or "sun". Here it is worth **diving deeper into the data collection process to make a more informed decision**.


In [ ]:
def replace_weather_strings(text):
    # Missing values appear as floats (`NaN`), so we leave them untouched for now.
    if not isinstance(text, float):
        # Normalize case and remove trailing punctuation first.
        text = text.lower().strip(".")

        # Map shorthand or inconsistent labels onto one standard category name.
        expressions = {
            r"\br\b": "rain",
            r"\bf\b": "fog",
            r"\b(sw|sn)\b": "snow",
            r"\bs\b": "sun",
            r"\b(d|driz)\b": "drizzle",
        }

        # Apply every regex replacement rule to the string.
        for key, value in expressions.items():
            text = re.sub(key, value, text)

    return text

In [ ]:
# Apply the cleaning function to every row in the `weather` column.
seattle_weather_data_pandas.weather = seattle_weather_data_pandas.weather.apply(
    lambda x: replace_weather_strings(x)
)

In [ ]:
seattle_weather_data_pandas["weather"].unique()

Let's check if every column is the type we are expecting:


In [ ]:
# `.info()` is a quick schema check: it shows data types and missing-value counts.
seattle_weather_data_pandas.info()

We see that `precipitation` is a **string** but we would expect a **float**, so we can use the `pd.to_numeric()` function to convert the values in that column to floats.  

If any values cannot be converted to **floats**, they will be replaced with **NaN**.  


In [ ]:
# Convert `precipitation` to numeric values.
# `errors="coerce"` means any non-convertible value becomes `NaN`.
seattle_weather_data_pandas["precipitation"] = pd.to_numeric(
    seattle_weather_data_pandas["precipitation"], errors="coerce"
)

In [ ]:
seattle_weather_data_pandas.info()

And last but not least, let's have a look at the `date` column. In Python there is a type `datetime`, so let's transform our data: 


In [ ]:
# Parse the date strings into pandas datetime objects.
# The explicit format tells pandas exactly how to read the values.
seattle_weather_data_pandas["date"] = pd.to_datetime(
    seattle_weather_data_pandas["date"], format="%Y/%m/%d"
)

In [ ]:
seattle_weather_data_pandas.info()

Now that we have the data in the format we want, let's **transform it**.  

First, we split our data to **simulate new incoming data**.  


In [ ]:
# Split the cleaned dataset into a training subset and a stand-in for future unseen data.
# `random_state=42` keeps the split reproducible for everyone.
train_weather_data, new_weather_data = train_test_split(
    seattle_weather_data_pandas, test_size=0.2, random_state=42
)

## Dealing with missing data

Missing data is common in real-world datasets. Typical causes include sensor failures, manual entry errors, system migrations and legacy collection processes.

Before choosing a strategy, inspect:
- how much data is missing,
- where the missing values occur,
- and whether missingness might carry information.

Common options:
- drop rows/columns (only when justified),
- impute with statistics (mean/median/mode),
- or use rule-based/domain-aware imputation.


### Visual Guide: Train-Based Imputation Reuse

```mermaid
flowchart TD
    A["Inspect missing values in practice_train"]
    B["Compute train medians and mode"]
    C["Store those train-derived values"]
    D["Impute practice_train"]
    E["Reuse the same values for practice_new"]

    A --> B --> C
    C --> D
    C --> E
```


> **Checkpoint:**
> You can identify missing-value patterns in both `train_weather_data` and `new_weather_data`.

> **Common pitfalls:**
> - Calculating imputation values on all data instead of train-only data.
> - Dropping rows too aggressively and changing dataset representativeness.
> - Assuming all missing values are random.

> **Self-check:**
> Can you justify *why* your chosen imputation strategy is appropriate for each column?


In [ ]:
# Count how many missing values appear in each column of the training set.
train_weather_data.isna().sum()

In [ ]:
# Convert missing-value counts into percentages so columns are easier to compare.
train_weather_data.isnull().mean() * 100

> **Note:** `.isna()` and `.isnull()` perform the exact same task. 


### Dropping data

**Dropping data** is the easiest and fastest option, but it is only recommended if there’s a **lot of data** to start with and the **percentage of missing values is low**. There are multiple ways to drop your data. You could **drop the rows** which have missing values in them or even **drop a whole column**. Though, you could end up with little data if there are too many missing values.  

Another problem is that you will most probably lose some (valuable) information. It could even happen that you **change the representation** of some data in cases when missing values have something in common and are not missing randomly. Hence you should always **be mindful about dropping data**.  

In our case we have only 3 columns with missing values.  


### Imputing data

Missing data and imputation are big topics in Data Science. Imputation means **replacing missing values with meaningful values**. In this notebook, we will work through several common imputation techniques.  

When you impute missing values, you are **manipulating the data**, which can have a **significant impact on its quality**. Hence you need to have a certain level of knowledge about your data in order to find a good approach which is best suited for the task at hand.  

As usual in Data Science, **there is no single solution** that fits every problem. Imputation ranges from fairly simple to highly complex strategies. Remember that Data Science work progresses in cycles. It is usually best to **start with a simple approach and increase complexity only when it adds value**. With this in mind, your preprocessing should only become more complex and more time-consuming if that extra work is justified.  

The first step is always to **determine how much missing data you have**, and don't forget: missing values are not necessarily denoted with NaNs.  

Since we already identified all our missing data above, we only have to use one command.  


#### Flag the missing value

Sometimes even a missing value can be informative. Let's say you ask survey participants their gender and only give the option of **female** or **male**. People who identify with a different gender may omit this question, resulting in a missing value. You could assign that value to `other` or another categorical value.  

In our case we could **assign a unique value**. For `precipitation` and `temperature` this would be a really **high absolute value**.  

#### Mean/Median/Mode imputation

One popular imputation technique is to use the **mean, median, or mode** of a column. The best choice depends on your data and the problem you are solving.  

For example, if you have a categorical value (like sex: 0 for male, 1 for female), taking the mean value would not make sense. Also, taking the mean for a highly skewed numerical value could be dangerous since **outliers could have a huge impact**. In that case, taking the median could be a better choice.  

Of course, it is also possible to make **assumptions**, for example "the weather today is the same as yesterday".  

> **Hint:** Visualizing your data with an appropriate plot could help you determine which value to take.  

As you probably already guessed, pandas already provides a built-in function for this task:  

`fillna` | Description
---|---
`df.fillna(0)` | Replaces all NaN in df with one value
`df.fillna({'precipitation': 0, 'temp_min': 0, 'temp_max': 42, 'wind': 0, 'weather': 'no_weather')}` | Replaces NaN in specific column with one specific value
`df.fillna(method='ffill')` | Use value of the day (row) before (forward fill)
`df.fillna(method='bfill')` | Use value of the day (row) after (backward fill)
`df.fillna(method='ffill', axis=1)` | Copies values from column before
`df.fillna(method='ffill', limit=1)` | If there are more than 1 value in a row missing, only the next one will be filled with the previous day value, the other one will stay as NaN


In [ ]:
train_weather_data.info()

In [ ]:
# Work on copies so we can compare imputed data with the original split if needed.
train_weather_data_pandas = train_weather_data.copy()
new_weather_data_pandas = new_weather_data.copy()

# Replace NaN in the numeric variables with the median.
for var in ["precipitation", "temp_max", "temp_min", "wind"]:
    # Learn the fill value from the training data only.
    # Using train-only statistics avoids leaking information from future data.
    value = train_weather_data_pandas[var].median()

    # Use the same learned value for both train and new data.
    train_weather_data_pandas[var] = train_weather_data_pandas[var].fillna(value)
    new_weather_data_pandas[var] = new_weather_data_pandas[var].fillna(value)

In [ ]:
# Replace missing values in the categorical column with the mode.
# `.mode()[0]` returns the most frequent category in the training set.
value = train_weather_data_pandas["weather"].mode()[0]

train_weather_data_pandas["weather"] = train_weather_data_pandas["weather"].fillna(
    value
)
new_weather_data_pandas["weather"] = new_weather_data_pandas["weather"].fillna(value)

In [ ]:
# Display number of missing values per column in the train data.
train_weather_data_pandas.isna().sum()

In [ ]:
# Display number of missing values per column in the new data.
new_weather_data_pandas.isna().sum()

## @TODO Practice 1 (Pandas): Missing-Value Audit + Imputation

**Exercise goal:** Practice a clean pandas workflow for auditing and imputing missing values.

@TODO:
1. Build a missing-value summary for `practice_train` with count + percentage.
2. Impute `precipitation` and `temp_max` with train medians.
3. Impute `weather` with the train mode.
4. Apply the same train-derived values to `practice_new`.

**Hints:**
- Keep the imputation statistics in named variables.
- Compute those values on `practice_train` only.
- Reuse the same values for `practice_new`.

> **Checkpoint:**
> `practice_train` and `practice_new` no longer have missing values in the selected columns.

Starter section: `01-feature-engineering-with-pandas.ipynb` (continue below)  
Reference walkthrough after your attempt: next cell


In [ ]:
# Exercise Starter (do this first)
# TODO: Complete the missing-value summary and imputation steps.

practice_train = train_weather_data.copy()
practice_new = new_weather_data.copy()

# @TODO 1: Create a summary DataFrame with missing counts and percentages.
# A small summary table is often the easiest way to audit missing data before imputing.
missing_summary = None

# @TODO 2-4: Impute `precipitation` and `temp_max` with train medians,
# and impute `weather` with the train mode.
# Keep all imputation values based only on training statistics.
# Reuse those same learned values on `practice_new`.

missing_summary, practice_train[["precipitation", "temp_max", "weather"]].isna().sum()

### Reference solution


In [ ]:
# Reference solution
practice_train_solution = train_weather_data.copy()
practice_new_solution = new_weather_data.copy()

# Build a compact audit table so we can see missing counts and percentages together.
missing_summary_solution = pd.DataFrame(
    {
        "missing_count": practice_train_solution.isna().sum(),
        "missing_pct": (practice_train_solution.isna().mean() * 100).round(2),
    }
)

# Impute numeric columns with the training median for each column.
num_cols = ["precipitation", "temp_max"]
for col in num_cols:
    fill_value = practice_train_solution[col].median()
    practice_train_solution[col] = practice_train_solution[col].fillna(fill_value)
    practice_new_solution[col] = practice_new_solution[col].fillna(fill_value)

# Impute the categorical column with the most frequent training category.
weather_mode = practice_train_solution["weather"].mode().iloc[0]
practice_train_solution["weather"] = practice_train_solution["weather"].fillna(
    weather_mode
)
practice_new_solution["weather"] = practice_new_solution["weather"].fillna(weather_mode)

(
    missing_summary_solution,
    practice_train_solution[["precipitation", "temp_max", "weather"]].isna().sum(),
)

Can you see a drawback of doing this only with pandas if you keep in mind that **you will need to apply the same transformation to incoming data later**?  

You would either need to recalculate the median/mean/mode every time new data arrives, or save those values somewhere and reuse them explicitly. This is where the **benefit of fitting a scikit-learn transformer** comes in (more on this in [notebook 02](../02-feature-engineering-with-sklearn-pipelines/02-feature-engineering-with-sklearn-pipelines.ipynb)).  

If we validate the DataFrame now with our Pydantic model from the top of the notebook, we don't get any errors anymore:  


In [ ]:
data_validation(train_weather_data_pandas, DataValidation)

## Further preprocessing steps

Of course, there are **further steps** you can or should take when preprocessing your data. Many algorithms, for example, need scaling or assume your data follows a normal distribution. Some algorithms can't handle strings in categories, so you have to encode the categories.  

In the following section you will come across a few examples for that.  


### Feature transformation of categorical features

There are different ways to **encode categories**. One of them is ordinal encoding. It turns out that this is not always a useful approach. Ordinal encoding, and also scikit-learn’s implementation of it, makes the fundamental assumption that numerical features reflect algebraic quantities. That means such a mapping would imply, for example, that **Queen Anne < Fremont < Wallingford**, or even that **Wallingford - Queen Anne = Fremont**, which (niche demographic jokes aside) does not make much sense.  

Therefore, **ordinal encoding only makes sense if you are dealing with an ordinal variable**. The difference to a nominal categorical variable is that **in ordinal variables there is a clear order in the categories**. Suppose you have a variable, e.g. **economic status** with three categories (low, medium and high). Even if the values are categories, they have a meaningful order.  

Whenever your variable lacks a meaningful order, a proven technique is to use **one-hot encoding**. Applying this technique to a categorical variable creates an extra column for each category, indicating the presence or absence of a category with the values 1 or 0. In pandas, you can do this with the `get_dummies()` function. You can decide to **drop one of the categories**. This is especially useful when perfectly collinear features cause problems, for example when feeding the encoded data into an unregularized linear regression model.  

However, **dropping one category breaks the symmetry of the original representation** and can therefore induce a bias in downstream models, for instance for penalized linear classification or regression models.  


In [ ]:
# One-hot encode the weather categories in the training data.
# `drop_first=True` removes one dummy column so the categories are not perfectly redundant.
train_weather_data_dummies = pd.get_dummies(train_weather_data.weather, drop_first=True)
train_weather_data_dummies

In [ ]:
# Apply the same one-hot encoding idea to the new data.
# The resulting columns may differ if a category is missing in one split.
new_weather_data_dummies = pd.get_dummies(new_weather_data.weather, drop_first=True)
new_weather_data_dummies

## @TODO Practice 2 (Pandas): Align One-Hot Encoded Data

**Exercise goal:** Make train/new one-hot encoded DataFrames consistent and model-ready.

@TODO:
1. One-hot encode `weather` for train and new data (`drop_first=True`).
2. Align encoded columns between train and new with `align(..., join='outer', fill_value=0)`.
3. Concatenate the encoded columns back to the non-categorical columns.

**Hints:**
- Encode train and new separately first.
- Use `train_encoded.align(new_encoded, join='outer', axis=1, fill_value=0)`.
- Verify that both outputs have matching columns in the same order.

> **Checkpoint:**
> Train and new feature matrices share the same encoded columns in the same order.

Starter section: `01-feature-engineering-with-pandas.ipynb` (continue below)  
Reference walkthrough after your attempt: next cell


In [ ]:
# Exercise Starter (do this first)
# TODO: Encode + align the train/new `weather` columns.

train_base = train_weather_data_pandas.copy()
new_base = new_weather_data_pandas.copy()

# @TODO 1: One-hot encode `weather` for train and new data.
# Keep the encoded outputs separate first so you can inspect their columns.
train_weather_ohe = None
new_weather_ohe = None

# @TODO 2: Align encoded columns with an outer join so both sets share identical feature columns.
# This avoids train/new mismatches if one split is missing a category.

# @TODO 3: Join encoded `weather` columns back to the remaining non-categorical columns.
train_model_matrix = None
new_model_matrix = None

train_model_matrix, new_model_matrix

### Reference solution


In [ ]:
# Reference solution
train_base_solution = train_weather_data_pandas.copy()
new_base_solution = new_weather_data_pandas.copy()

# Create dummy-variable columns for the categorical feature in each split.
train_weather_ohe_solution = pd.get_dummies(
    train_base_solution[["weather"]], drop_first=True
)
new_weather_ohe_solution = pd.get_dummies(
    new_base_solution[["weather"]], drop_first=True
)

# Align both encoded DataFrames so they contain the same columns in the same order.
# Missing columns are added and filled with 0.
train_weather_ohe_solution, new_weather_ohe_solution = train_weather_ohe_solution.align(
    new_weather_ohe_solution,
    join="outer",
    axis=1,
    fill_value=0,
)

# Combine the encoded weather columns back with the remaining features.
train_model_matrix_solution = pd.concat(
    [train_base_solution.drop(columns=["weather"]), train_weather_ohe_solution],
    axis=1,
)
new_model_matrix_solution = pd.concat(
    [new_base_solution.drop(columns=["weather"]), new_weather_ohe_solution],
    axis=1,
)

train_model_matrix_solution.head(), new_model_matrix_solution.head()

### Feature transformation of numerical features

Feature transformation is a process of transforming raw or unprocessed data into a more usable format that can be easily analyzed by machine learning models.  

The main goal of feature transformation is to **extract the most relevant information** from the data and **convert it into a more appropriate form** that can be used to train machine learning models. This transformation can involve various operations such as scaling, normalization, binning or encoding.  

When **scaling or normalizing** your data, you transform a feature by **converting its values to a particular range**, e.g. between 0 and 1. **Binning** is the process of **dividing continuous variables into discrete categories or bins**. We've already talked about **encoding** in the section before. In a nutshell, we're **converting categorical variables into numerical form** so our machine learning models can make use of them.  

Often the input features of your model have different units, which means that the variables also have different scales. While some model types (e.g. tree-based models like decision tree or random forest) are unaffected by the scale of numerical input variables, many machine learning algorithms, including algorithms using distance measures (e.g. KNN, SVM), perform better when the input features are **scaled to the same range**.  

Another challenge we often face is skewed data. **Skewness** is a measure of the **degree of asymmetry of the data**. When the values of a feature are not distributed symmetrically, that means they are not evenly distributed around the mean, we have to deal with skewed data.  

- A **positive skew** indicates that the data has a **longer tail on the right side**,  
- while a **negative skew** indicates that the data has a **longer tail on the left side**.  

Skewed data can pose challenges in machine learning as it can lead to **biased models**. In regression models, for example, skewed data can lead to the underestimation or overestimation of the dependent variable. Therefore, it is important to **transform skewed data before training a machine learning model**.  

A common technique for transforming skewed data is **log transformation**. This involves taking the natural logarithm of the data, which can help reduce skewness and make the distribution more symmetrical. Another technique is the **Box-Cox transformation**, which applies a power transformation to reduce skewness.  

Overall, **feature transformation is an important step in the machine learning pipeline**, as it helps to improve the accuracy of models by making the data more understandable and usable for the algorithms.  


In [ ]:
# Plot a histogram and a Q-Q plot for one variable.
# Histograms show the overall shape of the distribution,
# while Q-Q plots help us compare it to a normal distribution.
def diagnostic_plots(df, variable):
    """Function to plot a histogram and a Q-Q plot
    side by side, for a certain variable"""

    plt.figure(figsize=(15, 6))

    # Histogram: how the values are distributed across bins.
    plt.subplot(1, 2, 1)
    df[variable].hist(bins=30)

    # Q-Q plot: points close to the line suggest an approximately normal distribution.
    plt.subplot(1, 2, 2)
    stats.probplot(df[variable].astype(float), dist="norm", plot=plt)

    plt.show()

First let's have a look at the **distribution** of our features. With **histograms** and **QQ-Plots** you can check if they are more or less **normally distributed**.


In [ ]:
diagnostic_plots(train_weather_data_pandas, "precipitation")

`precipitation` is **heavily right skewed**, but most of the values are 0. In such cases, most methods for transformation don't work from scratch and there is **no universal way of dealing with it**. The problem is that a value of 0 can have different causes and each of them needs to be treated differently:  

- **Missing data**: Impute data / Drop observations if appropriate.  
- **Natural zero point** (e.g. time, precipitation, age)  
- **Sensitivity of measuring instrument**: Perhaps, add a small amount to data.  

In our case we will ignore it and simply **scale** it. Normally a data scientist would look into this issue during the EDA.  


In [ ]:
diagnostic_plots(train_weather_data_pandas, "temp_max")

In [ ]:
diagnostic_plots(train_weather_data_pandas, "temp_min")

The two `temperature` features are **nearly normally distributed**.


In [ ]:
diagnostic_plots(train_weather_data_pandas, "wind")

`wind` has a **slightly right-skewed** distribution.  

#### Transforming skewed data

Before we scale our data, we will **apply a transformation to get closer to a normal distribution**. There are quite a few transformations like Box-Cox, Yeo-Johnson, logarithmic, power, or square root transformation to name the most popular ones.  

We will use the **square root transformation** in this case:  


In [ ]:
# Apply a square-root transformation to reduce right skew in `wind`.
train_weather_data_pandas["wind"] = np.sqrt(train_weather_data_pandas["wind"])

In [ ]:
diagnostic_plots(train_weather_data_pandas, "wind")

After transforming, the `wind` feature looks **normally distributed**.  

#### Data standardization

Scaling the data can happen in different ways. The most commonly used techniques are **standardization** and **normalization**. Both techniques rescale our feature values.  

After applying **standardization** to a dataset, the values are rescaled in a way that the **mean of the transformed feature is 0 and the standard deviation is 1**. You can think of it as subtracting the mean value or centering the data. scikit-learn provides us with the `StandardScaler` for this case.  

In this notebook, we will create our own standardization function in order to scale the numerical features. As mentioned before, whether you apply scaling really depends on the model you choose in the end.  

A value is standardized as follows:  

$ x_{scaled} = \frac{x – \mu}{\sigma}  $, where  

- $ \mu = \frac{\sum{x}}{m} $ is the **mean**, where **m** is the **number of observations**  

- $ \sigma = \sqrt{ \frac{\sum{ (x – \mu)^2 }}{m}} $ is the **standard deviation**  


In [ ]:
# These are the columns we want to scale numerically.
numerical_features = ["precipitation", "temp_max", "temp_min", "wind"]


def standardize(df: pd.DataFrame, feature_name_lst: list[str]):
    # Standardization means: subtract the mean and divide by the standard deviation.
    for feature_name in feature_name_lst:
        x_mean = df[feature_name].mean()
        x_std = df[feature_name].std()

        # Apply the z-score formula column by column.
        df[feature_name] = df[feature_name].apply(lambda x: (x - x_mean) / x_std)

    # Return only the standardized feature columns so the assignment stays explicit.
    return df[feature_name_lst]


train_weather_data_pandas[numerical_features] = standardize(
    train_weather_data_pandas, numerical_features
)

In [ ]:
diagnostic_plots(train_weather_data_pandas, "temp_max")

Now the features are **centered around zero with a standard deviation of 1**. If we want to standardize new incoming data, we have to **use the mean and standard deviation of our known data**, so again we need to save this information in some way.  

> **Note:** All of the steps in this notebook were for showcasing and shouldn't be followed in reality. A good EDA beforehand is necessary to figure out how to deal with the data.  

In the next notebook you will see how we can **transform all of those steps into scikit-learn transformers and a pipeline**.  


### **Quiz**

1. **Why is feature engineering important in machine learning?**  
    [ ] To make datasets smaller  
    [ ] To ensure models can properly handle raw data representations  
    [ ] To automatically improve model interpretability  
    [ ] To remove the need for evaluation metrics  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** To ensure models can properly handle raw data representations  
      
      **Description:** Many models cannot handle missing values, categorical variables, or unscaled features directly. Feature engineering transforms data into formats that models can effectively process.
    </details>

---

2. **Which of the following models does *not* benefit from scaling (e.g. normalization or standardization)?**  
    [ ] Logistic Regression  
    [ ] Support Vector Machines  
    [ ] XGBoost  
    [ ] Linear Regression  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** XGBoost  
      
      **Description:** Tree-based models such as XGBoost or LightGBM are scale-invariant — scaling does not affect their performance.
    </details>

---

3. **What is a drawback of one-hot encoding ZIP codes directly?**  
    [ ] It creates too few categories  
    [ ] It reduces the dataset size  
    [ ] It can create tens of thousands of sparse features  
    [ ] It removes important location information  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** It can create tens of thousands of sparse features  
      
      **Description:** One-hot encoding high-cardinality variables like ZIP codes leads to massive, sparse datasets that are difficult for many models to handle.
    </details>

---

4. **Which of the following is a better feature engineering approach for ZIP codes than one-hot encoding?**  
    [ ] Dropping the ZIP code column  
    [ ] Aggregating into higher-level categories like city or state  
    [ ] Randomly assigning numerical values  
    [ ] Filling all values with the most common ZIP code  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** Aggregating into higher-level categories like city or state  
      
      **Description:** Grouping ZIP codes into meaningful aggregations preserves useful location information while reducing dimensionality.
    </details>

---

5. **In pandas, which code correctly fills missing values in the column `temp_max` with the column mean?**  
    [ ] `df['temp_max'].fillna(df.mean())`  
    [ ] `df['temp_max'] = df['temp_max'].fillna(df['temp_max'].mean())`  
    [ ] `df.fillna('temp_max'.mean())`  
    [ ] `df['temp_max'].replace(np.nan, df.mean())`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `df['temp_max'] = df['temp_max'].fillna(df['temp_max'].mean())`  
      
      **Description:** Missing values should be replaced with the column’s mean, not the entire DataFrame’s mean. The operation also needs assignment back to the column.
    </details>

---

6. **Given the DataFrame `df` with a `date` column of type string, which snippet converts it into a datetime type?**  
    [ ] `df['date'] = datetime(df['date'])`  
    [ ] `df['date'] = pd.to_datetime(df['date'])`  
    [ ] `df['date'] = df['date'].astype('datetime')`  
    [ ] `df['date'] = parse(df['date'])`  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** `df['date'] = pd.to_datetime(df['date'])`  
      
      **Description:** The `pd.to_datetime` function is the correct and robust way to convert string columns to datetime in pandas.
    </details>

---

7. **What is the role of pipelines in feature engineering?**  
    [ ] To combine models into a single algorithm  
    [ ] To automate and structure preprocessing steps  
    [ ] To train models faster by using parallel computing  
    [ ] To ensure categorical encoding is always one-hot  

    <details>
      <summary>Show Answer</summary>
      
      **Correct Answer:** To automate and structure preprocessing steps  
      
      **Description:** Pipelines allow transformations (e.g., scaling, encoding) to be applied systematically and consistently during training and inference.
    </details>
